# Adaptive-IPI: Standalone Pilot Validation


## Configuration
Configure parameters for the standalone pilot run. Change `TEACHER_MODEL` to swap the teacher.


In [ ]:
# Configuration parameters
TEACHER_MODEL = "Qwen/Qwen2.5-7B-Instruct"
TRAIN_SUBSET = 100
VALIDATION_SUBSET = 30
TEST_SUBSET = 100
USE_4BIT = True
USE_BFLOAT16 = True
RANDOM_SEED = 42
EXPERIMENT_NAME = "colab_pilot"



## 1. Install Dependencies


In [ ]:
!pip install -q torch transformers datasets accelerate bitsandbytes pandas numpy scikit-learn pyyaml matplotlib seaborn tqdm
!pip install -q flash-attn --no-build-isolation



## 2. Clone Repository


In [ ]:
import os
if not os.path.exists("Adaptive-IPI"):
    !git clone https://github.com/SubramaniBM/Adaptive-IPI.git
%cd Adaptive-IPI



## 3. Install Repository Requirements


In [ ]:
# Filter out vllm since it causes dependency conflicts in Colab (we use the transformers backend instead)
!grep -v "^vllm" requirements.txt > requirements_colab.txt
!pip install -r requirements_colab.txt



## 4. Verify Runtime


In [ ]:
import torch
import transformers
import psutil

print("="*40)
print("RUNTIME VERIFICATION")
print("="*40)

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    t = torch.cuda.get_device_properties(0).total_memory
    r = torch.cuda.memory_reserved(0)
    a = torch.cuda.memory_allocated(0)
    f = r - a  # free inside reserved
    print(f"Total VRAM: {t / 1e9:.2f} GB")
else:
    print("GPU NOT AVAILABLE. Please enable GPU in Colab (Runtime -> Change runtime type).")

print(f"Torch Version: {torch.__version__}")
print(f"Transformers Version: {transformers.__version__}")
print("="*40)
print(f"Teacher Model: {TEACHER_MODEL}")
if "32B" in TEACHER_MODEL and getattr(torch.cuda, "get_device_properties", None):
    if t / 1e9 < 30:
        print("WARNING: 32B model may not fit in memory with current GPU!")
else:
    print("Memory check passed for pilot model.")



## 5. Load Subset


In [ ]:
import pandas as pd
from pathlib import Path
import yaml
import json

# We create a new dataset_v99 for the colab pilot to strictly avoid modifying dataset_v0
dataset_dir = Path("data/processed/dataset_v0")
pilot_dir = Path("data/processed/dataset_v99")
pilot_dir.mkdir(parents=True, exist_ok=True)

for split, n in [("train", TRAIN_SUBSET), ("validation", VALIDATION_SUBSET), ("test", TEST_SUBSET)]:
    df = pd.read_csv(dataset_dir / f"{split}.csv")
    df = df.sample(n=min(n, len(df)), random_state=RANDOM_SEED)
    df.to_csv(pilot_dir / f"{split}.csv", index=False)
    print(f"Saved {len(df)} samples to {split}.csv")

# Create configs for pilot
student_config = {
    "model_id": "answerdotai/ModernBERT-base",
    "num_epochs": 1,
    "batch_size": 2,
    "learning_rate": 2.0e-5,
    "max_length": 512,
    "experiments_dir": "outputs/colab_experiments"
}
with open("configs/colab_student.yaml", "w") as f:
    yaml.dump(student_config, f)

teacher_config = {
    "model_id": TEACHER_MODEL,
    "backend": "transformers",
    "batch_size": 1,
    "backend_kwargs": {},
    "annotations_path": "data/processed/dataset_v99/teacher_annotations.jsonl"
}
with open("configs/colab_teacher.yaml", "w") as f:
    yaml.dump(teacher_config, f)

data_config = {
    "raw_dir": "data/raw",
    "processed_dir": "data/processed",
    "seed": RANDOM_SEED
}
with open("configs/colab_data.yaml", "w") as f:
    yaml.dump(data_config, f)

adaptive_config = {
    "generation": {
        "total_curriculum_samples": 10,
        "allocation_strategy": "proportional",
        "top_k_multiplier": 5,
        "benign_ratio": 0.5
    }
}
with open("configs/colab_adaptive.yaml", "w") as f:
    yaml.dump(adaptive_config, f)



## 6. Teacher Annotation


In [ ]:
!PYTHONPATH=. python scripts/run_phase2_teacher.py --config configs/colab_teacher.yaml --data-config configs/colab_data.yaml --dataset-version 99
print("✓ Teacher Annotation Complete")



## 7. Knowledge Distillation Training


In [ ]:
import subprocess

print("Running Phase 3: Distillation...")
res = subprocess.run(
    ["python", "scripts/run_phase3_distill.py", "--student-config", "configs/colab_student.yaml", "--data-config", "configs/colab_data.yaml", "--dataset-version", "99"],
    capture_output=True, text=True, env={"PYTHONPATH": "."}
)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError("Phase 3 failed.")

baseline_exp_dir = None
for line in res.stdout.split("\n") + res.stderr.split("\n"):
    if "Experiment:" in line:
        baseline_exp_dir = line.split("Experiment:")[1].strip()

if not baseline_exp_dir:
    raise RuntimeError("Could not find experiment directory in Phase 3 output.")
print(f"✓ Knowledge Distillation Complete. Saved to: {baseline_exp_dir}")



## 8. Evaluate


In [ ]:
print("Running Phase 4: Evaluation...")
checkpoint_path = f"{baseline_exp_dir}/checkpoint/best"
res = subprocess.run(
    ["python", "scripts/run_phase4_analyze.py", "--checkpoint", checkpoint_path, "--data-config", "configs/colab_data.yaml", "--dataset-version", "99"],
    capture_output=True, text=True, env={"PYTHONPATH": "."}
)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError("Phase 4 failed.")

failure_report_path = None
for line in res.stdout.split("\n") + res.stderr.split("\n"):
    if "Report:" in line:
        failure_report_path = line.split("Report:")[1].strip() + "/failures_validation.json"

if not failure_report_path:
    raise RuntimeError("Could not find failure report in Phase 4 output.")
print(f"✓ Evaluation & Profiling Complete. Report: {failure_report_path}")



## 9. Failure Profiling


In [ ]:
# Handled within Phase 4 output.
print("✓ Failure Profiling Complete")



## 10. Curriculum Generation


In [ ]:
print("Running Phase 5: Curriculum Generation...")
res = subprocess.run(
    ["python", "scripts/run_phase5_generate.py", "--teacher-config", "configs/colab_teacher.yaml", "--adaptive-config", "configs/colab_adaptive.yaml", "--data-config", "configs/colab_data.yaml", "--dataset-version", "99", "--failure-report", failure_report_path],
    capture_output=True, text=True, env={"PYTHONPATH": "."}
)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError("Phase 5 failed.")

curriculum_path = None
for line in res.stdout.split("\n") + res.stderr.split("\n"):
    if "curriculum saved to" in line.lower():
        curriculum_path = line.lower().split("to")[1].strip()

print("✓ Curriculum Generation Complete.")



## 11. Retraining


In [ ]:
print("Running Phase 6: Retraining...")
res = subprocess.run(
    ["python", "scripts/run_phase6_retrain.py", "--student-config", "configs/colab_student.yaml", "--data-config", "configs/colab_data.yaml", "--dataset-version", "99"],
    capture_output=True, text=True, env={"PYTHONPATH": "."}
)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError("Phase 6 failed.")

retrained_exp_dir = None
for line in res.stdout.split("\n") + res.stderr.split("\n"):
    if "Experiment:" in line:
        retrained_exp_dir = line.split("Experiment:")[1].strip()

if not retrained_exp_dir:
    raise RuntimeError("Could not find retrained experiment directory in Phase 6 output.")
print(f"✓ Retraining Complete. Saved to: {retrained_exp_dir}")



## 12. Final Evaluation


In [ ]:
print("Running Phase 7: Final Evaluation...")
retrained_checkpoint = f"{retrained_exp_dir}/checkpoint/best"
res = subprocess.run(
    ["python", "scripts/run_phase7_evaluate.py", "--checkpoint", retrained_checkpoint, "--data-config", "configs/colab_data.yaml", "--dataset-version", "100"],
    capture_output=True, text=True, env={"PYTHONPATH": "."}
)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError("Phase 7 failed.")

print("✓ Final Evaluation Complete")



## Results Comparison


In [ ]:
import json
import os

def load_metrics(exp_dir):
    try:
        with open(f"{exp_dir}/metrics.json") as f:
            return json.load(f)
    except Exception as e:
        print(f"Could not load metrics from {exp_dir}: {e}")
        return {}

baseline = load_metrics(baseline_exp_dir) if 'baseline_exp_dir' in locals() else {}
retrained = load_metrics(retrained_exp_dir) if 'retrained_exp_dir' in locals() else {}

print("================================================================")
print(f"{'Metric':<15} | {'Baseline (KD)':<20} | {'KD + Adaptive (Retrained)'}")
print("----------------------------------------------------------------")
for metric in ['accuracy', 'precision', 'recall', 'f1', 'auroc', 'ece']:
    b_val = baseline.get(metric, 0)
    r_val = retrained.get(metric, 0)
    print(f"{metric:<15} | {b_val:<20.4f} | {r_val:.4f}")
print("================================================================")

